# Day K5 — 멀티모달 모델: 환자 단위 검증, 결측 모달, 그리고 센서 기여도를 묻는 올바른 방법

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/K-multimodal-fog/day-K5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/K-multimodal-fog/day-K5.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 모달별 표를 window 키로 합치기

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch

fs = 500
KEY = ["rec", "start"]                  # K2의 window 키(합성 데이터에는 task가 없음)

def window_starts(n, fs=500, win_s=2.0, step_s=1.0):    # K2의 함수: 2초 window, 50% 겹침
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, s)

def window_label(y_win, rule="majority"):               # K2의 함수
    if rule == "majority": return int(y_win.mean() >= 0.5)
    if rule == "any":      return int(y_win.max() == 1)
    if rule == "center":   return int(y_win[len(y_win) // 2])

def skeleton(seed=0, fog=(25, 35), kind="tremble"):
    """K1의 합성 뼈대를 함수로 감쌌다. 기본값이면 뼈대와 똑같다. kind="akinetic"은 떨림 없는 동결(가정)."""
    t = np.arange(0, 60, 1/fs)
    label = ((t >= fog[0]) & (t < fog[1])).astype(int)
    rng = np.random.default_rng(seed)
    frozen = 0.4*np.sin(2*np.pi*5.0*t) if kind == "tremble" else 0.05*np.sin(2*np.pi*1.0*t)
    acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), frozen) + 0.1*rng.standard_normal(t.size)
    return t, label, acc, rng

def fi(x):
    f, p = welch(x, fs=fs, nperseg=len(x))
    return p[(f >= 3) & (f < 8)].sum() / p[(f >= 0.5) & (f < 3)].sum()

# 합성 환자 8명(P8은 두 세션). 모든 FOG 관련 모양은 설명용 가정이다.
plan = [("P1", 1, (25, 35), "tremble"), ("P2", 2, (10, 20), "tremble"),
        ("P3", 3, (30, 45), "akinetic"), ("P4", 4, (40, 50), "tremble"),
        ("P5", 5, (0, 0), "tremble"),                      # FOG가 한 번도 없는 환자
        ("P6", 6, (20, 26), "tremble"), ("P7", 7, (15, 30), "akinetic"),
        ("P8/OFF_1", 8, (25, 35), "tremble"), ("P8/OFF_2", 9, (35, 50), "akinetic")]
acc_t, emg_t, eeg_t, sc_t, lab_t = [], [], [], [], []
for rec, seed, fog, kind in plan:
    t, label, acc, rng = skeleton(seed, fog, kind)
    gain = rng.uniform(0.5, 2.0)                         # 환자마다 다른 근전도 증폭(단위 없음)
    eeg_offset = rng.normal(0, 0.03)                     # 환자마다 다른 EEG 기저값
    sc_base = rng.uniform(500, 2000)                     # 환자마다 다른 피부전도 기저값(ADC 카운트, FOG와 무관하게 둠)
    for s in window_starts(t.size):
        w = slice(s, s + 2 * fs)
        k = {"rec": rec, "start": int(s)}
        y = window_label(label[w])
        lab_t.append({**k, "fog": y})
        acc_t.append({**k, "acc_fi": fi(acc[w]), "acc_sd": acc[w].std()})
        if rec not in ("P2", "P6"):                      # 두 환자는 EMG 센서가 없었다(가정)
            emg_t.append({**k, "emg_rms": gain * (1.6 if y else 1.0) * rng.lognormal(0, 0.25)})
        eeg_t.append({**k, "eeg_rel_theta": 0.15 + eeg_offset + 0.02 * y + rng.normal(0, 0.03)})
        sc_t.append({**k, "sc_mean": sc_base + rng.normal(0, 20)})
lab, acc_f, emg_f, eeg_f, sc_f = (pd.DataFrame(r) for r in (lab_t, acc_t, emg_t, eeg_t, sc_t))
print("행 수  label %d | acc %d | emg %d | eeg %d | sc %d" % (len(lab), len(acc_f), len(emg_f), len(eeg_f), len(sc_f)))

feat = (lab.merge(acc_f, on=KEY, how="left", validate="one_to_one")
           .merge(emg_f, on=KEY, how="left", validate="one_to_one")
           .merge(eeg_f, on=KEY, how="left", validate="one_to_one")
           .merge(sc_f, on=KEY, how="left", validate="one_to_one"))
feat["patient"] = feat["rec"].str[:2]                  # 기록 → 환자 (P8의 두 세션은 같은 환자)
print("합친 표:", feat.shape)
print(feat.groupby("patient").agg(windows=("fog", "size"), fog=("fog", "sum"),
                                  emg_missing=("emg_rms", lambda v: v.isna().mean())).to_string())
inner = lab.merge(emg_f, on=KEY, how="inner")
print("how='inner'로 합치면 남는 행:", len(inner), "| 사라지는 환자:", sorted(set(lab.rec) - set(inner.rec)))

### 예시 2 — 환자 단위 LeaveOneGroupOut과, FOG 없는 환자의 fold

예시 1에 이어서 실행합니다.

In [ ]:
import warnings
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

FEATS = ["acc_fi", "acc_sd", "emg_rms", "eeg_rel_theta", "sc_mean"]
X, y, groups = feat[FEATS], feat["fog"].to_numpy(), feat["patient"].to_numpy()   # group = 환자(세션 아님)

oof = np.zeros(len(y))
fold_ap = {}
for tr, te in LeaveOneGroupOut().split(X, y, groups):
    model = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[tr], y[tr])   # NaN을 그대로 받는다
    oof[te] = model.predict_proba(X.iloc[te])[:, 1]
    with warnings.catch_warnings(record=True) as wlist:
        warnings.simplefilter("always")
        fold_ap[groups[te][0]] = average_precision_score(y[te], oof[te])
        if wlist:
            print("경고(", groups[te][0], "):", str(wlist[0].message)[:60])
print("fold별 PR-AUC:", {k: round(v, 3) for k, v in fold_ap.items()})
print("fold 평균 PR-AUC: %.3f  ← P5(FOG 0개)의 0.000이 섞여 있다" % np.mean(list(fold_ap.values())))
print("pooled PR-AUC: %.3f | 기준값(FOG 비율) %.3f" % (average_precision_score(y, oof), y.mean()))
p5 = groups == "P5"
print("P5: window %d개 중 확률 0.5 이상(오경보) %d개" % (p5.sum(), (oof[p5] >= 0.5).sum()))

### 예시 3 — 결측 처리, 두 모델, 그리고 세 가지 기여도

예시 1에 이어서 실행합니다(예시 2를 건너뛰어도 됨). random forest는 window 531개 × fold 8개를 여러 번 학습하므로 20초쯤 걸립니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

MODAL = {"acc": ["acc_fi", "acc_sd"], "emg": ["emg_rms"], "eeg": ["eeg_rel_theta"], "sc": ["sc_mean"]}
y, groups = feat["fog"].to_numpy(), feat["patient"].to_numpy()

def make(kind):
    if kind == "RF":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=0)
    return HistGradientBoostingClassifier(random_state=0)

def logo_oof(cols, kind="HGB", data=feat):
    """환자 하나씩 빼고 학습 → 뺀 환자 예측. 모든 fold의 예측을 모아(pooled) 돌려준다."""
    X, yy, gg = data[cols], data["fog"].to_numpy(), data["patient"].to_numpy()
    oof = np.zeros(len(yy))
    for tr, te in LeaveOneGroupOut().split(X, yy, gg):
        oof[te] = make(kind).fit(X.iloc[tr], yy[tr]).predict_proba(X.iloc[te])[:, 1]
    return oof

ALL = sum(MODAL.values(), [])
# (1) 결측 모달 처리 세 가지 (HGB)
opts = {"그대로 NaN (모델이 처리)": feat,
        "EMG 없는 환자 빼기": feat[feat["emg_rms"].notna()]}
for name, data in opts.items():
    o = logo_oof(ALL, "HGB", data)
    print(f"{name:22s} 환자 {data.patient.nunique()}명  PR-AUC {average_precision_score(data.fog, o):.3f}")
o = logo_oof([c for c in ALL if c != "emg_rms"], "HGB")
print(f"{'EMG를 모두에게서 빼기':22s} 환자 {feat.patient.nunique()}명  PR-AUC {average_precision_score(y, o):.3f}")

# (2) 모델 두 종류
for kind in ["RF", "HGB"]:
    print(kind, "pooled PR-AUC %.3f" % average_precision_score(y, logo_oof(ALL, kind)))

# (3) 모달 제거(ablation): 그 모달을 빼고 '다시 학습'해서 떨어지는 만큼
full = average_precision_score(y, logo_oof(ALL, "RF"))
for m, cols in MODAL.items():
    ap = average_precision_score(y, logo_oof([c for c in ALL if c not in cols], "RF"))
    print(f"ablation  -{m:3s}: PR-AUC {ap:.3f} (변화 {ap - full:+.3f})")

# (4) impurity importance: 학습 자료 안에서 갈라 쓴 정도 (전체로 한 번 학습)
rf = make("RF").fit(feat[ALL], y)
imp = pd.Series(rf.feature_importances_, index=ALL)
print("impurity  :", {m: float(round(imp[cols].sum(), 3)) for m, cols in MODAL.items()})

# (5) permutation importance(모달 단위): 학습된 모델은 그대로 두고, test fold에서 그 모달의 열을 함께 섞는다
rng = np.random.default_rng(0)
perm = {m: np.zeros((5, len(y))) for m in MODAL}          # 5번 반복 × window
for tr, te in LeaveOneGroupOut().split(feat, y, groups):
    model = make("RF").fit(feat[ALL].iloc[tr], y[tr])
    Xte = feat[ALL].iloc[te]
    for m, cols in MODAL.items():
        for r in range(5):
            Xp = Xte.copy()
            Xp[cols] = Xte[cols].to_numpy()[rng.permutation(len(te))]   # 같은 순서로 함께 섞어 모달 안의 관계는 유지
            perm[m][r, te] = model.predict_proba(Xp)[:, 1]
for m in MODAL:
    d = [full - average_precision_score(y, perm[m][r]) for r in range(5)]
    print(f"permutation {m:3s}: PR-AUC 감소 {np.mean(d):.3f} ± {np.std(d):.3f}")

## 직접 해보기

**문제 1 — 안전하게 합치기.** 빈칸을 채워 라벨 표에 EMG 표를 붙이세요. 행 수가 531로 유지되어야 합니다.

In [ ]:
merged = lab.merge(emg_f, on=____, how="left", validate="one_to_one")
print(merged.shape, "| emg 결측 행:", int(merged["emg_rms"].isna().sum()))

In [ ]:
merged = lab.merge(emg_f, on=KEY, how="left", validate="one_to_one")

**문제 2 — group 고르기.** 빈칸을 채워 fold가 환자 수(8)만큼 나오게 하세요.

In [ ]:
groups = feat[____].to_numpy()
print("fold 수:", LeaveOneGroupOut().get_n_splits(groups=groups))

In [ ]:
groups = feat["patient"].to_numpy()

**문제 3 — 환자별 보고.** 예시 3의 함수로 pooled 예측을 만든 뒤, FOG가 있는 환자는 PR-AUC와 기준값을, 없는 환자는 오경보율을 출력하세요.

In [ ]:
o_pat = logo_oof(ALL, "RF")
for pid, g in feat.assign(p=o_pat).groupby("patient"):
    yy, pp = g["fog"].to_numpy(), g["p"].to_numpy()
    if ____:
        print(f"{pid}: PR-AUC {average_precision_score(yy, pp):.3f} (기준값 {yy.mean():.2f})")
    else:
        print(f"{pid}: FOG 없음 → 오경보율 {(pp >= 0.5).mean():.2f}")

In [ ]:
    if yy.sum() > 0:

**문제 4 — 모달 단위로 섞기.** 예시 3의 (5)에서 한 모달의 열을 **같은 순서로 함께** 섞는 줄입니다. 빈칸을 채우세요.

In [ ]:
Xp[cols] = Xte[cols].to_numpy()[____]

In [ ]:
Xp[cols] = Xte[cols].to_numpy()[rng.permutation(len(te))]

**문제 5 — 두 모달을 함께 빼기.** 빈칸을 채워 ACC, EMG, ACC+EMG를 뺀 세 경우를 비교하세요.

In [ ]:
for drop in [["acc"], ["emg"], ["acc", "emg"]]:
    cols = [c for m, cs in MODAL.items() if ____ for c in cs]
    print("-%-8s PR-AUC %.3f" % ("+".join(drop), average_precision_score(y, logo_oof(cols, "RF"))))

In [ ]:
    cols = [c for m, cs in MODAL.items() if m not in drop for c in cs]

### 실제 데이터로 해보기 (선택)

**1. 모달별 특징 표를 만들고 window 키로 합치기**

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from scipy.signal import welch, butter, sosfiltfilt

DATA = "data/xuanwu/Filtered Data"
FS, WIN = 500, 1000                                      # 2초 window
KEY = ["rec", "task", "start"]                          # K2의 window 키 (rec = "001" … 또는 "008/OFF_1")
EEG = ["FP1","FP2","F3","F4","C3","C4","P3","P4","O1","O2","F7","F8","P7","P8",
       "Fz","Cz","Pz","FC1","FC2","CP1","CP2","FC5","FC6","CP5","CP6"]
IMU = [f"{s}_{q}" for s in ["LShank","RShank","Waist","Arm"] for q in ["ax","ay","az","gx","gy","gz","x7"]]
COLS = ["idx","clock"] + EEG + ["c27","c28","c29","c30","c31"] + IMU + ["label"]   # K1의 열 이름
# README 표의 전경골근(TA) 열 번호 (K1)
L_TA = {"001":28,"002":28,"003":27,"004":27,"005":27,"006":28,"007":28,"008/OFF_1":28,"008/OFF_2":31,"009":27,"010":27,"011":27,"012":27}
R_TA = {"001":27,"002":27,"003":28,"004":28,"005":28,"006":27,"007":27,"008/OFF_1":27,"008/OFF_2":27,"009":28,"010":28,"011":28,"012":28}

def window_starts(n, fs=500, win_s=2.0, step_s=1.0):    # K2의 함수: 2초 window, 50% 겹침
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, s)

def window_label(y_win, rule="majority"):               # K2의 함수
    if rule == "majority": return int(y_win.mean() >= 0.5)
    if rule == "any":      return int(y_win.max() == 1)
    if rule == "center":   return int(y_win[len(y_win) // 2])

def band(x, lo, hi, nperseg):
    f, p = welch(x, fs=FS, nperseg=nperseg, axis=-1)
    return p[..., (f >= lo) & (f < hi)].sum(-1)

acc_rows, emg_rows, sc_rows, eeg_rows, lab_rows, used = [], [], [], [], [], []
for path in sorted(glob.glob(os.path.join(DATA, "**", "task_*.txt"), recursive=True)):
    parts = os.path.relpath(path, DATA).split(os.sep)
    rec, task = "/".join(parts[:-1]), parts[-1][:-4]     # "003" 또는 "008/OFF_1", "task_1"
    d = pd.read_csv(path, header=None, names=COLS, usecols=COLS[2:])   # 행 번호·시각 문자열은 빼고 읽는다
    side = "LShank" if (d[["LShank_ax", "LShank_ay", "LShank_az"]] != 0).any(axis=None) else "RShank"   # 왼쪽이 없으면 오른쪽
    ta = [f"c{L_TA[rec]}", f"c{R_TA[rec]}"]                # 양쪽 전경골근
    acc_mag = np.sqrt(sum(d[f"{side}_{q}"].to_numpy() ** 2 for q in ["ax", "ay", "az"]))
    sc = d["Arm_x7"].to_numpy()                          # 팔 IMU의 7번째 열 = SC
    has_sc = np.any(sc != 0)
    eeg = d[["Cz", "FC1", "FC2"]].to_numpy().T
    sos = butter(4, (20, 240), btype="bandpass", fs=FS, output="sos")   # K3: 파일의 EMG는 10 Hz 아래 파워가 대부분
    emg = {c: sosfiltfilt(sos, d[c].to_numpy() - d[c].mean()) for c in ta}
    y = d["label"].to_numpy().astype(int)
    used.append({"rec": rec, "shank": side, "SC": has_sc})
    for s in window_starts(len(d)):
        k = dict(rec=rec, task=task, start=int(s))
        w = slice(s, s + WIN)
        x = acc_mag[w] - acc_mag[w].mean()
        acc_rows.append({**k, "acc_sd": x.std(),
                         "acc_fi": band(x, 3, 8, WIN) / band(x, 0.5, 3, WIN)})
        emg_rows.append({**k, "emg_ta_logrms": np.mean([np.log(np.sqrt(np.mean(emg[c][w] ** 2))) for c in ta])})
        if has_sc:                                        # 센서가 없는 기록은 행 자체를 만들지 않는다
            tt = np.arange(WIN) / FS
            sc_rows.append({**k, "sc_mean": sc[w].mean(), "sc_slope": np.polyfit(tt, sc[w], 1)[0]})
        tot = band(eeg[:, w], 1, 45, 500)
        eeg_rows.append({**k, "eeg_rel_theta": (band(eeg[:, w], 4, 8, 500) / tot).mean(),
                         "eeg_rel_beta": (band(eeg[:, w], 13, 30, 500) / tot).mean(),
                         "eeg_hi_60_100": np.log(band(eeg[:, w], 60, 100, 500).mean())})
        lab_rows.append({**k, "fog": window_label(y[w], "majority")})

print(pd.DataFrame(used).drop_duplicates().to_string(index=False))
tables = {name: pd.DataFrame(r) for name, r in
          [("label", lab_rows), ("acc", acc_rows), ("emg", emg_rows), ("sc", sc_rows), ("eeg", eeg_rows)]}
for name, tb in tables.items():
    print(f"{name:5s} 행 {len(tb):6d}  키 중복 {tb.duplicated(KEY).sum()}")
feat = tables["label"]
for name in ["acc", "emg", "sc", "eeg"]:
    feat = feat.merge(tables[name], on=KEY, how="left", validate="one_to_one")   # 라벨 표 기준으로 붙인다
feat.to_csv("features_K5.csv", index=False)
print("합친 표:", feat.shape, "| SC 결측 window:", int(feat["sc_mean"].isna().sum()))

**2. 환자 단위 검증, 환자별 보고, ablation, 두 가지 importance**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

feat = pd.read_csv("features_K5.csv", dtype={"rec": str})   # "001"의 0을 지키려면 str
feat["patient"] = feat["rec"].str[:3]                   # 기록 → 환자 (008의 두 세션은 한 환자)
MODAL = {"acc": ["acc_sd", "acc_fi"], "emg": ["emg_ta_logrms"], "sc": ["sc_mean", "sc_slope"],
         "eeg": ["eeg_rel_theta", "eeg_rel_beta", "eeg_hi_60_100"]}
ALL = sum(MODAL.values(), [])
y = feat["fog"].to_numpy()

def make(kind):
    if kind == "RF":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)
    return HistGradientBoostingClassifier(random_state=0)

def logo_oof(cols, kind="RF", groups=feat["patient"].to_numpy()):
    oof = np.zeros(len(y))
    for tr, te in LeaveOneGroupOut().split(feat, y, groups):
        oof[te] = make(kind).fit(feat[cols].iloc[tr], y[tr]).predict_proba(feat[cols].iloc[te])[:, 1]
    return oof

print("window %d개, 환자 %d명, 세션 %d개, FOG 비율(PR-AUC 기준값) %.3f"
      % (len(feat), feat.patient.nunique(), feat["rec"].nunique(), y.mean()))
res = {}
for kind in ["RF", "HGB"]:
    res[kind] = logo_oof(ALL, kind)
    print(f"{kind:3s} pooled PR-AUC {average_precision_score(y, res[kind]):.3f}  ROC-AUC {roc_auc_score(y, res[kind]):.3f}")

# 환자별 보고 (RF)
rows = []
for pid, g in feat.assign(p=res["RF"]).groupby("patient"):
    yy, pp = g["fog"].to_numpy(), g["p"].to_numpy()
    rows.append({"patient": pid, "windows": len(g), "fog_%": round(100 * yy.mean(), 1),
                 "PR_AUC": round(average_precision_score(yy, pp), 3) if yy.sum() >= 1 else "정의 안 됨",
                 "sens@0.5": round((pp[yy == 1] >= 0.5).mean(), 2) if yy.sum() else "-",
                 "false_alarm@0.5": round((pp[yy == 0] >= 0.5).mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

# group을 세션으로 잡으면? (008의 두 세션이 서로를 돕는다)
ses_groups = feat["rec"].to_numpy()
o_ses = logo_oof(ALL, "RF", ses_groups)
m8 = (feat["patient"] == "008").to_numpy()
print("008 PR-AUC: group=환자 %.3f | group=세션 %.3f" % (average_precision_score(y[m8], res["RF"][m8]),
                                                       average_precision_score(y[m8], o_ses[m8])))

# 모달 제거(ablation): 빼고 다시 학습
full = average_precision_score(y, res["RF"])
for m, cols in MODAL.items():
    ap = average_precision_score(y, logo_oof([c for c in ALL if c not in cols]))
    print(f"ablation -{m:3s}: PR-AUC {ap:.3f} ({ap - full:+.3f})")
for m, cols in MODAL.items():
    ap = average_precision_score(y, logo_oof(cols))
    print(f"{m:3s}만 사용  : PR-AUC {ap:.3f}")

# impurity importance (전체로 한 번 학습, 모달별 합)
imp = pd.Series(make("RF").fit(feat[ALL], y).feature_importances_, index=ALL)
print("impurity:", {m: float(round(imp[c].sum(), 3)) for m, c in MODAL.items()})

# permutation importance (모달 단위, test fold 안에서 섞기, 5회)
rng = np.random.default_rng(0)
perm = {m: np.zeros((5, len(y))) for m in MODAL}
for tr, te in LeaveOneGroupOut().split(feat, y, feat["patient"]):
    model = make("RF").fit(feat[ALL].iloc[tr], y[tr])
    Xte = feat[ALL].iloc[te]
    for m, cols in MODAL.items():
        for r in range(5):
            Xp = Xte.copy()
            Xp[cols] = Xte[cols].to_numpy()[rng.permutation(len(te))]
            perm[m][r, te] = model.predict_proba(Xp)[:, 1]
for m in MODAL:
    d = [full - average_precision_score(y, perm[m][r]) for r in range(5)]
    print(f"permutation {m:3s}: PR-AUC 감소 {np.mean(d):.3f} ± {np.std(d):.3f}")